In [ ]:
from modules.utils import merge_devtest_csvs
from modules.viz import ConfigLookup, metric_x_point, test_summary
from pathlib import Path
from pandas import DataFrame # typing
import pandas as pd
import shutil


In [ ]:
class MultiExperiment():
    def __init__(
        self,
        experiment_dirs: str | list[str|Path],
        keys: str | list[str],
        parent_dir: str | Path = './',
        out_dir: str | Path = './experiments',
    ):
        self.parent_dir = Path(parent_dir)
        self.keys = keys if isinstance(keys, list) else [keys]
        self.out_dir = Path(out_dir)

        # make output directory
        self.out_dir.mkdir(parents=True, exist_ok=True)

        # ensure list and paths are children of parent
        if not isinstance(experiment_dirs, list):
            experiment_dirs = [experiment_dirs]
        self.experiment_dirs = [self._child_path(Path(exp_dir)) for exp_dir in experiment_dirs]

        # merge devtest csvs
        self.dev: DataFrame = self._merge_devtests('dev.csv')
        self.test: DataFrame = self._merge_devtests('test.csv')
        self.configs: list[str] = self.test['config'].unique().tolist()

        # get configs
        # self.conf_lookup = ConfigLookup(
        #     keys=self.keys,
        #     path=self.parent_dir,
        #     configs=self.configs
        # )

    def _child_path(self, path: Path) -> Path:
        # path already child: parent / (child - parent)
        try:
            return self.parent_dir / path.relative_to(self.parent_dir)
        
        # path not child: parent / path
        except ValueError:
            raise self.parent_dir / path

    def _merge_devtests(self, file: str) -> DataFrame:
        expt_dfs: list[DataFrame] = []
        old_max = -1 # start at -1 + 1 = 0

        for expt_dir in self.experiment_dirs:
            df = pd.read_csv(expt_dir/file)

            # align trials
            new_min =  df['trial'].min() # min of current expt
            shift = (old_max + 1) - new_min # shift to align with old max + 1
            df['trial'] += shift # shift trials to start after old max
            old_max = df['trial'].max() # update old max for next expt

            expt_dfs.append(df)

        return pd.concat(expt_dfs, ignore_index=True)
    
    def _copy_fn(self, src: str, dst: str) -> str:
        return dst if Path(dst).exists() else shutil.copy2(src, dst)

    def _copy_expt_files(self):
        # make config directory
        self.config_dir = self.out_dir / 'configs'
        self.config_dir.mkdir(parents=True, exist_ok=True)

        # copy expt files into config directory
        for expt in self.experiment_dirs:
            expt = expt.resolve()
            if not expt.is_dir():
                continue # skip non-directories

            shutil.copytree(
                src=str(expt),
                dst=str(self.config_dir / expt.name),
                dirs_exist_ok=True,
                copy_function=self._copy_fn
            )

        

